In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


PATH = "raw_loan_dataset (1).csv"
df = pd.read_csv(PATH)
print(df.shape)

(103, 7)


In [24]:
print(df.head())
print(df.info())

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasColla

In [25]:
#missing value counts
print(df.isnull().sum())

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


In [26]:
#removeing $ ,  income column
df['Income'] = df['Income'].replace(r"[\$,]", "", regex=True).astype(float)
#removeing $ ,  LoanAmount column
df['LoanAmount'] = df['LoanAmount'].replace(r"[\$,]", "", regex=True).astype(float)
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0             Y   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  


In [27]:
print("HasCollateral categorical error ")
print(df['HasCollateral'].value_counts(dropna=False))

print("PreviousDefaults  categorical error ")
print(df['PreviousDefaults'].value_counts(dropna=False))

print("Approved categorical error ")
print(df['Approved'].value_counts(dropna=False))

HasCollateral categorical error 
HasCollateral
No     50
Yes    46
Y       2
NaN     2
N       1
yse     1
yes     1
Name: count, dtype: int64
PreviousDefaults  categorical error 
PreviousDefaults
No     84
Yes    13
NaN     2
1       1
0       1
Y       1
N       1
Name: count, dtype: int64
Approved categorical error 
Approved
Yes         65
No          32
Approved     1
Rejected     1
approved     1
rejected     1
YES          1
NO           1
Name: count, dtype: int64


In [28]:
# 3 Fix Category Errors before Imputation
yes_no_map = {
    "Y": "Yes", "YES": "Yes", "yse": "Yes", "1": "Yes", "yes": "Yes", "Approved": "Yes", "approved": "Yes", "y": "Yes",
    "N": "No", "NO": "No", "0": "No", "no": "No", "rejected": "No", "Rejected": "No", "n": "No" 
}
df['HasCollateral'] = df['HasCollateral'].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df['Approved'] = df['Approved'].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})


In [29]:
#after
print(" After HasCollateral  ")
print(df['HasCollateral'].value_counts(dropna=False))

print(" After PreviousDefaults ")
print(df['PreviousDefaults'].value_counts(dropna=False))

print(" After Approved  ")
print(df['Approved'].value_counts(dropna=False))

 After HasCollateral  
HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64
 After PreviousDefaults 
PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64
 After Approved  
Approved
Yes    68
No     35
Name: count, dtype: int64


In [30]:
 # 4 Impute Missing Values
df['Income'] = df['Income'].fillna(df['Income'].median())
df['CreditScore'] = df['CreditScore'].fillna(df['CreditScore'].median())
df['EmploymentYears'] = df['EmploymentYears'].fillna(df['EmploymentYears'].median())
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['HasCollateral'] = df['HasCollateral'].fillna(df['HasCollateral'].mode()[0])
df['PreviousDefaults'] = df['PreviousDefaults'].fillna(df['PreviousDefaults'].mode()[0])

In [31]:
print("after:  ", "\n", df.isnull().sum())
print(df.shape)

after:   
 Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64
(103, 7)


In [32]:
# 5 Remove Duplicates
before = df.shape[0]
df = df.drop_duplicates()
print(f"before: {before} \nafter: {df.shape[0]} rows ")


before: 103 
after: 100 rows 


In [33]:
# 6 Outliers (IQR capping)
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])   

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)

In [34]:
# 7 Label Encoding
# approved column
label_map = {"Yes": 1, "No": 0}
df['Approved'] = df['Approved'].map(label_map).astype(int)
print("after label encoding:  ")
print(df['Approved'].value_counts())
print(df['Approved'].value_counts(normalize=True).round(3))

after label encoding:  
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [35]:
# PreviousDefaults column
df['PreviousDefaults'] = df['PreviousDefaults'].map(label_map).astype(int)
print("after label encoding ")
print(df['PreviousDefaults'].value_counts())

after label encoding 
PreviousDefaults
0    85
1    15
Name: count, dtype: int64


In [36]:
# HasCollateral  
df['HasCollateral'] = df['HasCollateral'].map(label_map).astype(int)
print("after label encoding ")
print(df['PreviousDefaults'].value_counts())

after label encoding 
PreviousDefaults
0    85
1    15
Name: count, dtype: int64


In [37]:
print(df["Approved"].unique())

[0 1]


In [38]:
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")


Class balance OK for baseline Accuracy (both classes well represented).


In [39]:
print("after label encoding:  ")
print(df['Approved'].value_counts())
print(df['Approved'].value_counts(normalize=True).round(3))

after label encoding:  
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [40]:
# 9 Feature Engineering
df['DebtToIncome'] = df['LoanAmount'] / df['Income'].replace(0, np.nan)
df['IncomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1)
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0              1   
1   96482.0        524.0             15.0     11200.0              1   
2   28478.0        638.0              5.4     12100.0              0   
3   25851.0        561.0             17.6      7000.0              1   
4   38396.0        527.0              9.8     10700.0              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.237111           51814.285714  
1                 0         1      0.116084            6030.125000  
2                 0         1      0.424889            4449.687500  
3                 0         1      0.270783            1389.838710  
4                 0         1      0.278675            3555.185185  


In [41]:
# 10 Feature Scaling (Research & Choose)
binary_cols = {"HasCollateral", "PreviousDefaults", "Approved"}
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]
scaler = MinMaxScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])
print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.585174     0.121560         0.024490    0.339800              1   
1  0.498215     0.091743         0.591837    0.101287              1   
2  0.018530     0.353211         0.200000    0.115989              0   
3  0.000000     0.176606         0.697959    0.032673              1   
4  0.088490     0.098624         0.379592    0.093118              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.258294               0.846035  
1                 0         1      0.104832               0.082051  
2                 0         1      0.496399               0.055679  
3                 0         1      0.300991               0.004620  
4                 0         1      0.310998               0.040752  


In [52]:
# 11 Final Checks & Save
print("====final info====","\n", df.info())
print("====final missing values=====","\n", df.isnull().sum())
print("====final head======", "\n", df.head())

CLEAN_DATA = "clean_loan_dataset.csv"
df.to_csv(CLEAN_DATA, index=False)
print("saved cleaned data for ", CLEAN_DATA)


<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB
====final info==== 
 None
====final missing values===== 
 Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
Income